# Molecular dynamics analysis of HIV-1 protease

This Jupyter notebook presents a molecular dynamics analysis of the Human Immunodeficiency Virus type I (HIV-1) protease. We simulated HIV-1 protease (PDB ID: 1HVR) without its ligand for 201 ns. Then, a total of 201 frames were extracted at regular intervals of 1 ns from the molecular dynamics’ trajectory.

Here, we describe the conformational changes of a cavity that defines the active site of the HIV-1 protease, which is an effective therapeutic target. The HIV-1 protease catalytic cycle involves movements of β -hairpins, called 'flaps', which control the accessibility of substrates to the active site of the homodimer. Further, we performed a occurence of cavity points, that were detected in at least two frames, and we plotted all properties (volume, area, depth and hydropathy) throughout the simulation.

In [1]:
# Import required modules
import os
import numpy
import pickle
import pyKVFinder
import KVFinderMD
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering

## HIV-1 protease trajectory

## Cavity detection and characterization workflow


In [2]:
%%timeit -r 3 -n 1
# Create KVFinderMD object
md = KVFinderMD.KVFinderMD()

# Load HIV-1 protease trajectory
md.read_trajectory("data/HIV.pdb")

# Custom detection parameters
probe_out = 12.0
volume_cutoff = 50.0

# Perform detection and characterization
md.detect(
    # Analysis modes
    analyze_constitutional=True, 
    export_occurrence=True,
    analyze_spatial=True, 
    analyze_depth=True, 
    analyze_hydropathy=True,
    # Custom parameters
    probe_out=probe_out, 
    volume_cutoff=volume_cutoff,
    # Miscellaneous
    basedir='results/spde',
    verbose=True
)

# Write characterization to file
md.write('results/spde/results.toml')

# Save md with pickle
with open("results/spde/md.pkl", "wb") as f:
    pickle.dump(md, f)

[Frame:   0]  Elapsed time: 00s  Estimated time: 25s            

/home/ABTLUS/joao.guerra/remote-repos/jvsguerra/IV-CEC/.venv/lib/python3.10/site-packages/MDAnalysis/core/universe.py:743: UserWarning: Reader has no dt information, set to 1.0 ps
  dt=self.trajectory.ts.dt * step,



[Frame:   1]  Elapsed time: 02s  Estimated time: 04m03s            
[Frame:   2]  Elapsed time: 04s  Estimated time: 05m11s            
[Frame:   3]  Elapsed time: 07s  Estimated time: 05m46s            
[Frame:   4]  Elapsed time: 09s  Estimated time: 06m03s            
[Frame:   5]  Elapsed time: 11s  Estimated time: 06m16s            
[Frame:   6]  Elapsed time: 13s  Estimated time: 06m26s            
[Frame:   7]  Elapsed time: 16s  Estimated time: 06m31s            
[Frame:   8]  Elapsed time: 18s  Estimated time: 06m34s            
[Frame:   9]  Elapsed time: 20s  Estimated time: 06m36s            
[Frame:  10]  Elapsed time: 23s  Estimated time: 06m37s            
[Frame:  11]  Elapsed time: 25s  Estimated time: 06m39s            
[Frame:  12]  Elapsed time: 27s  Estimated time: 06m39s            
[Frame:  13]  Elapsed time: 29s  Estimated time: 06m40s            
[Frame:  14]  Elapsed time: 32s  Estimated time: 06m40s            
[Frame:  15]  Elapsed time: 34s  Estimated time

## Cavity alignment analysis

We explored three formulations for structural alignment of cavities:
- 3D grid alignment;
- (2D) Contact matrix alignment;
- (2D) Distance matrix alignment - Inspired on DALI structural alignment.


In [3]:
# Create directories
os.makedirs("results/spde/grid", exist_ok=True)
os.makedirs("results/spde/contact", exist_ok=True)
os.makedirs("results/spde/distance", exist_ok=True)

### 3D grid alignment: clustering 3D grid of each cavity detected throughout the molecular dynamics simulation

Here, we separate each cavity in a different boolean grid (1: cavity; 0: everything else).

In [4]:
if os.path.exists("results/spde/md.pkl"):
    with open("results/spde/md.pkl", "rb") as f:
        md = pickle.load(f)
    md.occurrence._gap = 1

In [5]:
# Separate each cavity in a different boolean grid
cavities = list()
frames = list()

for n in range(md.n_frames):
    f = md.frame(n)
    frames.append(n)
    
    for ncav in range(f.n_cavities):
        cavities.append(f.cavities == ncav+2)
        frames.append(f)

cavities = numpy.asarray(cavities)
frames = numpy.asarray(frames)

# Show grids
print(cavities.shape)

(672, 160, 126, 105)


In [6]:
# Prepare data
# NOTE: Memory consuming step
cavities = cavities.astype(bool).reshape(672, -1)

# Show data
print(cavities.shape)

(672, 2116800)


#### Agglomerative clustering based on Silhouette Scores

In [7]:
%%timeit -r 3 -n 1
preds = AgglomerativeClustering(n_clusters=10, metric='correlation', linkage="complete").fit_predict(cavities)

4min 46s ± 2.24 s per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [8]:
preds = AgglomerativeClustering(n_clusters=10, metric='correlation', linkage="complete").fit_predict(cavities)

KVFinderMD.silhouette(
    cavities,
    preds,
    metric='dice',
    filename=f'results/spde/grid/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * preds[i]
    pyKVFinder.export(
        f'results/spde/grid/{preds[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None,
        md.kvtraj._vertices,
        md.kvtraj._step,
        B=B
    )

### (2D) Contact matrices alignment: clustering contact matrix of each cavity detected throughout the molecular dynamics simulation

Here, we define a contact matrix for each cavity, considering the interface residues surrounding it.

#### Explore distance metrics in 2D comparison of contact matrices

The distance metrics are used to assess the similarity of the adjacency matrices, ie the detected cavities.

The standard metrics: 
- Euclidean distance
- Correlation

Pairwise distances for booleans:
- Dice dissimilarity
- Hamming distance
- Jaccard dissimilarity
- Rogers-Tanimoto dissimilarity
- Russell-Rao dissimilarity
- Sokal-Michener dissimilarity
- Sokal-Sneath dissimilarity
- Yule dissimilarity

Discussion about those metrics:
- https://www.ibm.com/docs/en/spss-statistics/SaaS?topic=measures-distances-similarity-binary-data
- https://stats.stackexchange.com/questions/61705/similarity-coefficients-for-binary-data-why-choose-jaccard-over-russell-and-rao

In [9]:
# Create CavityAlignment object
alignment = KVFinderMD.CavityAlignment(md)

In [10]:
# Explore contact matrix alignment
alignment.explore(method="contact")

In [11]:
alignment.scores

,Number of Clusters,Silhouette Score
Correlation,11.0,0.557755
Dice,16.0,0.524279
Hamming,2.0,0.622585
Jaccard,16.0,0.401248
Rogers-Tanimoto,2.0,0.621012
Rusell-Rao,8.0,0.001496
Sokal-Michener,2.0,0.621012
Sokal-Sneath,16.0,0.279258
Yule,16.0,0.811657


In [12]:
%%timeit -r 3 -n 1
# Alignment
alignment.align(method='contact', affinity="dice")

5.03 s ± 50.8 ms per loop (mean ± std. dev. of 3 runs, 1 loop each)


In [13]:
alignment.align(method='contact', affinity="dice")

# Plot silhouette
KVFinderMD.silhouette(
    alignment.contacts.reshape(672, -1),
    alignment.clusters,
    metric='dice',
    filename=f'results/spde/contact/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * alignment.clusters[i]
    pyKVFinder.export(
        f'results/spde/contact/{alignment.clusters[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None, 
        md.kvtraj._vertices, 
        md.kvtraj._step,
        B=B
    )

/home/ABTLUS/joao.guerra/remote-repos/jvsguerra/IV-CEC/.venv/lib/python3.10/site-packages/sklearn/metrics/pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)
/home/ABTLUS/joao.guerra/remote-repos/jvsguerra/IV-CEC/.venv/lib/python3.10/site-packages/sklearn/metrics/pairwise.py:2462: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)


### (2D) Distance matrices alignment: clustering contact matrix of each cavity detected throughout the molecular dynamics simulation

In [14]:
# Explore contact matrix alignment
alignment.explore(method="distance")

In [15]:
alignment.scores

,Number of Clusters,Silhouette Score
Euclidean,15.0,0.569840
Correlation,15.0,0.525135
Bray-Curtis,15.0,0.524059
Canberra,3.0,0.728975
Chebyshev,2.0,0.455468
City Block,8.0,0.714505
Cosine,15.0,0.528961
Jensen-Shannon,12.0,0.363735
Minkowski,15.0,0.569840
Squared Euclidean,15.0,0.723011


In [16]:
%%timeit -r 5 -n 3
# Alignment
alignment.align(method='distance', affinity="cosine")

4.24 s ± 17 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [17]:
alignment.align(method='distance', affinity="cosine")

# Plot silhouette
KVFinderMD.silhouette(
    alignment.contacts.reshape(672, -1),
    alignment.clusters,
    metric="cosine",
    filename=f'results/spde/distance/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * alignment.clusters[i]
    pyKVFinder.export(
        f'results/spde/distance/{alignment.clusters[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None, 
        md.kvtraj._vertices, 
        md.kvtraj._step,
        B=B
    )